## LangChain Agent with tools

In [1]:
import importlib
import json

import src.agent as agent_module
import src.agent_helpers.models as models_module
import src.agent_helpers.search_execution as search_execution_module
import src.agent_helpers.search_graph as search_graph_module
import src.config as config_module
import src.resolution_cache as cache_module


importlib.reload(models_module)
importlib.reload(search_execution_module)
importlib.reload(search_graph_module)
importlib.reload(agent_module)
importlib.reload(config_module)
importlib.reload(cache_module)

from src.agent import Agent
from src.agent_helpers.models import DocumentContext
from src.resolution_cache import ResolutionCacheService, ResolutionCacheWriter


agent_config = config_module.load_agent_config()
agent = Agent(agent_config)
cache_writer = (
    ResolutionCacheWriter(agent_config.cache_path)
    if agent_config.cache_path is not None
    else None
)
resolver = ResolutionCacheService(agent, cache_writer)


async def find_link_to_document(
    reference: str,
    document_context: DocumentContext | None = None,
    *,
    doc_reference: str | None = None,
):
    """Resolve one extracted legal reference and persist cacheable responses."""
    return await resolver.resolve(
        reference,
        document_context=document_context,
        doc_reference=doc_reference,
    )


def format_resolution(response, full: bool = False):
    """Print a readable summary followed by the complete response structure."""
    print("=" * 80)
    print("RESOLUTION SUMMARY")
    print("=" * 80)
    print(f"Reference : {response.reference}")
    print(f"Document Class : {response.classification.document_type} [{response.classification.jurisdiction}]")
    print(f"Status    : {response.status.value}")
    print(f"Reason    : {response.reason}")
    print(f"Attempts  : {response.attempt_count}")
    print(f"Results   : {len(response.results)}")
    

    for index, result in enumerate(response.results, start=1):
        print(f"\nResult {index}")
        print(f"  Title       : {result.title}")
        print(f"  Source      : {result.source}")
        print(f"  Document ID : {result.doc_id}")
        print(f"  URL         : {result.url}")
        print(f"  Retrieved   : {result.retrieved_at.isoformat()}")
        print(f"  Metadata    : {result.metadata}")

    if response.proposed_guideline_lesson:
        print(f"\nProposed lesson: {response.proposed_guideline_lesson}")

    if full:
        print("\n" + "=" * 80)
        print("FULL RESPONSE STRUCTURE")
        print("=" * 80)
        print(json.dumps(response.model_dump(mode="json"), indent=2, ensure_ascii=False))

c:\Users\StefaanVercoutere\anaconda3\envs\abb\Lib\site-packages\fastmcp\server\auth\providers\jwt.py:10: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import JsonWebKey, JsonWebToken


## Document-level cache experiment

This notebook cache is read-only. It indexes the newest eligible JSONL record by canonical document reference, ignores passage locators, and falls back to the existing resolver when document identity is uncertain.

In [2]:
from datetime import datetime, timezone
from pathlib import Path
import json
import re

from src.agent_helpers.models import ResolutionResponse
from src.resolution_cache import is_cacheable


STOPWORDS = {
    "and", "bij", "de", "der", "een", "en", "het", "of", "on", "op",
    "te", "the", "tot", "van", "voor",
}


def normalize_reference(value) -> str:
    return re.sub(r"\s+", " ", str(value or "").strip()).casefold()


def title_tokens(value) -> set[str]:
    return {
        token
        for token in re.findall(r"\w+", normalize_reference(value))
        if len(token) > 2 and token not in STOPWORDS
    }


def _record_timestamp(record: dict) -> datetime:
    try:
        stamp = datetime.fromisoformat(str(record.get("created_at")).replace("Z", "+00:00"))
    except (TypeError, ValueError):
        return datetime.min.replace(tzinfo=timezone.utc)
    return stamp if stamp.tzinfo else stamp.replace(tzinfo=timezone.utc)


def _is_eligible(record: dict) -> bool:
    response = record.get("response") or {}
    if response.get("status") != "resolved":
        return False
    if len(response.get("results") or []) != 1:
        return False
    return all(
        attempt.get("complete", True)
        and not attempt.get("truncated", False)
        and not attempt.get("source_errors")
        for attempt in response.get("trace", [])
    )


def load_document_cache(cache_path: Path | None = None) -> dict:
    """Index the newest eligible resolved record by canonical document reference."""
    path = Path(cache_path or agent_config.cache_path or "resolution-cache.jsonl")
    index: dict[str, dict] = {}
    if not path.exists():
        return index
    with path.open(encoding="utf-8") as cache_file:
        for line in cache_file:
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            doc_reference = record.get("doc_reference")
            if not doc_reference or not _is_eligible(record):
                continue
            key = normalize_reference(doc_reference)
            if key not in index or _record_timestamp(record) >= _record_timestamp(index[key]):
                index[key] = record
    return index


def find_cache_record(doc_reference: str) -> tuple[dict | None, str]:
    """Look up a cached record by exact reference, then by fuzzy title overlap."""
    key = normalize_reference(doc_reference)
    if key in document_cache:
        return document_cache[key], "exact_match"

    reference_tokens = title_tokens(doc_reference)
    if len(reference_tokens) < 2:
        return None, "no_match"

    # Deduplicate by result identity so duplicate entries for the same
    # document do not read as ambiguous.
    matches: dict[tuple, dict] = {}
    for record in document_cache.values():
        response = record.get("response") or {}
        result = (response.get("results") or [{}])[0]
        titles = ((response.get("classification") or {}).get("title"), result.get("title"))
        if any(
            len(tokens := title_tokens(title)) >= 2 and tokens <= reference_tokens
            for title in titles
        ):
            identity = (result.get("source"), result.get("doc_id") or result.get("url"))
            matches[identity] = record

    records = list(matches.values())
    if len(records) == 1:
        return records[0], "title_match"
    if len(records) > 1:
        return None, "ambiguous_match"
    return None, "no_match"


document_cache = load_document_cache()
print(f"Loaded {len(document_cache)} document-level cache entries.")


async def find_link_to_document_cached(
    reference: str,
    document_context: DocumentContext | None = None,
    *,
    doc_reference: str | None = None,
    reference_class: str | None = None,
    verbose: bool = True,
):
    canonical_reference = doc_reference or reference
    record, lookup = find_cache_record(canonical_reference)
    if record is not None:
        try:
            cached_response = ResolutionResponse.model_validate(record["response"])
        except Exception as error:
            if verbose:
                print(f"[CACHE REJECTED] {canonical_reference!r} lookup={lookup} error={error}")
        else:
            if verbose:
                print(f"[CACHE HIT] {canonical_reference!r} lookup={lookup}")
            return cached_response, {"status": "hit", "lookup": lookup}
    else:
        if verbose:
            print(f"[CACHE MISS] {canonical_reference!r} lookup={lookup}")

    response = await find_link_to_document(
        canonical_reference,
        document_context=document_context,
        doc_reference=doc_reference,
    )
    if is_cacheable(response):
        new_record = {
            "doc_reference": canonical_reference,
            "response": response.model_dump(mode="json"),
            "created_at": datetime.now(timezone.utc).isoformat(),
        }
        document_cache[normalize_reference(canonical_reference)] = new_record
    return response, {"status": "miss", "lookup": lookup}


Loaded 247 document-level cache entries.


# extracting and marking mentions in the text

In [3]:
from datetime import datetime, timezone
from pathlib import Path
from dotenv import dotenv_values
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from bs4 import BeautifulSoup
from src.agent_helpers.models import ResolutionResponse
from src.models import SearchResult
import re
import requests


class DocumentReference(BaseModel):
    doc_reference: str
    reference_class: str
    context: str
    mentions: list[str]


class DocumentReferences(BaseModel):
    references: list[DocumentReference]


def create_llm():
    env = dotenv_values(".env")
    return ChatOpenAI(
        api_key=env["subscription_key"],
        base_url=env["azure_endpoint"],
        model=env["deployment"],
    )


def fetch_formatted_page(url: str) -> tuple[str, str]:
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup.select("script, noscript"):
        tag.decompose()

    html = str(soup)

    text_soup = BeautifulSoup(html, "html.parser")
    for tag in text_soup.select("style, nav, footer, svg"):
        tag.decompose()
    for item in text_soup.select("li"):
        item.insert(0, "• ")

    lines = [line.strip() for line in text_soup.get_text("\n").splitlines()]
    text = "\n".join(line for line in lines if line)
    return html, text


REFERENCE_STYLE = """
<style>
a.legal-reference {
    border-radius: 0.15em;
    color: inherit;
    padding: 0 0.15em;
    text-decoration: underline;
    text-decoration-style: dotted;
}
a.legal-reference.resolved {
    background-color: #c8f7c5;
}
a.legal-reference.unresolved {
    background-color: #fff3a3;
}
</style>
"""


def _reference_and_response(item):
    if isinstance(item, dict) and "reference" in item:
        return item["reference"], item.get("results")
    return item, None


def mark_mention_in_soup(content, mention: str, attrs: dict, soup: BeautifulSoup) -> bool:
    """Mark one mention substring as an <a class='legal-reference'> tag in the soup.

    Handles both single-text-node occurrences and mentions that cross multiple
    consecutive inline tags or spans (e.g. Aspose.Words styling artifacts in ebesluitvorming).
    """
    if not mention or not mention.strip():
        return False
    mention = mention.strip()
    pattern = "".join(
        r"\s+" if part.isspace() else re.escape(part)
        for part in re.split(r"(\s+)", mention)
    )
    try:
        matcher = re.compile(pattern, re.IGNORECASE)
    except re.error:
        matcher = re.compile(re.escape(mention), re.IGNORECASE)

    # 1. Fast path: single text node
    for node in list(content.find_all(string=True)):
        if node.parent and node.parent.name in {"a", "script", "style"}:
            continue
        source = str(node)
        match = matcher.search(source)
        if match:
            ref_tag = soup.new_tag("a", attrs=attrs.copy())
            ref_tag.string = source[match.start():match.end()]
            node.replace_with(source[:match.start()], ref_tag, source[match.end():])
            return True

    # 2. Multi-node container path (handles mentions crossing consecutive spans/tags)
    for container in content.find_all(["p", "li", "h1", "h2", "h3", "h4", "h5", "h6", "td", "th", "div"]):
        if container.find(["p", "li", "h1", "h2", "h3", "h4", "h5", "h6", "td", "th"]):
            continue
        nodes = []
        full_text = ""
        for s in container.find_all(string=True):
            if s.parent and s.parent.name in {"a", "script", "style"}:
                continue
            start = len(full_text)
            full_text += str(s)
            end = len(full_text)
            nodes.append((s, start, end))
        if not nodes:
            continue
        m = matcher.search(full_text)
        if not m:
            continue
        overlap = [n for n in nodes if n[2] > m.start() and n[1] < m.end()]
        if not overlap:
            continue

        matched_text = full_text[m.start():m.end()]
        ref_tag = soup.new_tag("a", attrs=attrs.copy())
        ref_tag.string = matched_text

        if len(overlap) == 1:
            s, start, _ = overlap[0]
            s.replace_with(str(s)[:m.start() - start], ref_tag, str(s)[m.end() - start:])
            return True

        first_s, first_start, _ = overlap[0]
        last_s, last_start, _ = overlap[-1]
        middle = overlap[1:-1]
        before_text = str(first_s)[:m.start() - first_start]
        after_text = str(last_s)[m.end() - last_start:]

        anc_first = first_s
        while anc_first.parent and anc_first.parent != container:
            anc_first = anc_first.parent
        anc_last = last_s
        while anc_last.parent and anc_last.parent != container:
            anc_last = anc_last.parent

        anc_first.insert_after(ref_tag)
        first_s.replace_with(before_text)
        last_s.replace_with(after_text)
        for mid_s, _, _ in middle:
            p = mid_s.parent
            mid_s.extract()
            if p and p != container and not p.get_text():
                p.extract()
        if anc_first != container and not anc_first.get_text():
            anc_first.extract()
        if anc_last != container and not anc_last.get_text():
            anc_last.extract()
        return True

    return False


def write_linked_html(html: str, references: list, output_file) -> None:
    soup = BeautifulSoup(html, "html.parser")
    content = soup.body or soup

    for item in references:
        reference, response = _reference_and_response(item)
        resolved = (
            response is not None
            and response.status.value == "resolved"
            and response.results
        )
        result_url = response.results[0].url if resolved else None
        link_class = "resolved" if result_url else "unresolved"
        attrs = {
            "class": f"legal-reference {link_class}",
            "data-doc-reference": reference.doc_reference,
            "data-reference-class": reference.reference_class,
            "data-context": reference.context,
            "data-resolution-status": (
                response.status.value if response is not None else "unresolved"
            ),
        }
        if result_url:
            attrs.update({
                "href": result_url,
                "target": "_blank",
                "rel": "noopener noreferrer",
            })

        marked = False
        for mention in reference.mentions:
            if mark_mention_in_soup(content, mention, attrs, soup):
                marked = True
                break

        if not marked:
            # Fallback attempts for slight punctuation or formatting differences
            fallbacks = []
            chars_to_strip = " '" + '".,;:()'
            for mention in reference.mentions:
                stripped = mention.strip(chars_to_strip)
                if stripped and stripped != mention:
                    fallbacks.append(stripped)
            # Distinctive OMV identifier fallback
            omv_match = re.search(r"OMV_\d{10}[A-Za-z0-9_]*", reference.doc_reference)
            if omv_match:
                fallbacks.append(omv_match.group(0))
            # Distinctive spatial plan name fallback (e.g. 'ANTWERPSESTEENWEG - ORCHIDEESTRAAT')
            cleaned_ref = reference.doc_reference.replace('"', "'")
            rup_match = re.search(r"(?:RUP|ruimtelijk uitvoeringsplan)\s+[']?([A-Z0-9\s\-]+)[']?", cleaned_ref, re.IGNORECASE)
            if rup_match:
                plan_name = rup_match.group(1).strip()
                if len(plan_name) > 3:
                    fallbacks.append(plan_name)

            for fb in fallbacks:
                if mark_mention_in_soup(content, fb, attrs, soup):
                    break

    if soup.head:
        soup.head.append(BeautifulSoup(REFERENCE_STYLE, "html.parser"))
    else:
        soup.insert(0, BeautifulSoup(REFERENCE_STYLE, "html.parser"))

    output_file.write_text(str(soup), encoding="utf-8")


def read_linked_html(input_file) -> tuple[str, list[dict]]:
    html = Path(input_file).read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "html.parser")
    content = soup.body or soup

    processed: list[dict] = []
    for link in content.select("a.legal-reference"):
        doc_reference = link.get("data-doc-reference") or link.get_text(strip=True)
        reference_class = link.get("data-reference-class", "")
        context = link.get("data-context", "")
        resolution_status = link.get("data-resolution-status", "unresolved")
        mention = str(link.string or "").strip()
        result_url = link.get("href")

        existing = next(
            (
                item for item in processed
                if item["reference"].doc_reference == doc_reference
            ),
            None,
        )
        if existing:
            existing["reference"].mentions.append(mention)
            continue

        reference = DocumentReference(
            doc_reference=doc_reference,
            reference_class=reference_class,
            context=context,
            mentions=[mention],
        )
        response = ResolutionResponse(
            reference=doc_reference,
            status=resolution_status,
            results=[
                SearchResult(
                    title=doc_reference,
                    url=result_url,
                    source="linked_html",
                    doc_id=result_url or doc_reference,
                    retrieved_at=datetime.now(timezone.utc),
                    metadata={"cached_from": "linked_html"},
                )
            ] if result_url else [],
            reason="Cached from linked HTML",
            attempt_count=1,
            trace=[],
        )
        processed.append({"reference": reference, "results": response})

    return html, processed

REFERENCE_EXTRACTION_PROMPT = """
Find references to specific legislative, regulatory, and administrative planning or permit documents in the input and
organize them by the underlying document they refer to.

Return one group for each distinct underlying document, even when the document is
mentioned several times with different articles, paragraphs, sections, titles, or
other citation details. Do not resolve or merge different instruments merely
because they have similar names.

For each group return:
- doc_reference: a concise canonical reference to the underlying instrument,
  including enough title, date, number, jurisdiction, or approval information to
  search for that instrument once;
- reference_class: the document type, never a topic or legal effect;
- context: a concise description of the subject or use of this instrument in the
  input document, including relevant project, permit, planning, construction,
  environmental, education, or other domain context;
- mentions: every exact occurrence in the input that belongs to this document.

Each mention must be a verbatim substring of the input. Preserve capitalization,
accents, articles, whitespace, quotation marks, punctuation, and exact wording.
Keep article, paragraph, title, and section details in the mention even though
those details do not create a new document group. Do not return a surrounding
sentence, rewrite text, merge separate documents, invent a title, or omit a
mention because another mention identifies the same instrument.

A reference must identify an instrument that could be searched as its own
document. If the text is only a generic rule, duty, generic permission or prohibition, definition,
deadline, remedy, condition, advice, or generic legal wording without an identifiable instrument or permit, do not extract it.
A sentence containing a legal word is not automatically a reference.

Relevant instruments include:
- named decrees, laws, constitutions, codes, royal or government decrees, ministerial decisions;
- named municipal regulations or police regulations;
- named European regulations and directives;
- spatial plans (ruimtelijk uitvoeringsplan / RUP, bijzonder plan van aanleg / BPA, gewestplan);
- formal standards and building regulations (bouwreglement / stedenbouwkundige verordening);
- environmental permits, permit applications, and permit decisions (omgevingsvergunning,
  stedenbouwkundige vergunning, milieuvergunning), specifically including their official
  identifiers such as OMV dossier numbers (e.g. "OMV_2026070977", "OMV_2020011285") or
  decision numbers (e.g. CBS, CVB, or GR decisions). When an OMV identifier is mentioned,
  extract it as a document reference so it can be resolved.
For generic names such as "politiereglement", "reglement", "wet", or "besluit",
include title qualifiers and adjacent date or approval context in the mention.
Never return a date on its own.
"""


async def extract_references(url: str, llm=None) -> dict:
    html, text = fetch_formatted_page(url)
    llm = llm or create_llm()
    extractor = llm.with_structured_output(DocumentReferences)
    result = await extractor.ainvoke([
        SystemMessage(content=REFERENCE_EXTRACTION_PROMPT),
        HumanMessage(content=text),
    ])

    # Ensure any OMV dossier identifiers in the document are captured even if omitted by LLM
    existing_mentions = {
        m.lower()
        for ref in result.references
        for m in ref.mentions
    }
    omv_matches = re.findall(r"\b(OMV_\d{10}[A-Za-z0-9_]*)\b", text)
    for omv_id in dict.fromkeys(omv_matches):
        if not any(omv_id.lower() in m for m in existing_mentions):
            result.references.append(
                DocumentReference(
                    doc_reference=f"Omgevingsvergunning {omv_id}",
                    reference_class="omgevingsvergunning",
                    context=f"Omgevingsvergunning dossier {omv_id}",
                    mentions=[omv_id],
                )
            )
            existing_mentions.add(omv_id.lower())

    return {
        "html": html,
        "text": text,
        "references": result.references,
    }


## scraping of the agandapunten from ebesluitvorming.gent.be

In [7]:
from src.tools.ebesluitvorming import agenda_item_url
import asyncio
import re
from typing import Any
import httpx
from bs4 import BeautifulSoup
from pathlib import Path


BASE_URL = "https://ebesluitvorming.gent.be"
_MEETING_ID_RE = re.compile(r"(?:/zittingen/|^)([0-9]{2}\.[0-9]{4}\.[0-9]{4}\.[0-9]{4})")
_DOCUMENT_ID_RE = re.compile(r"\b\d{4}_\w+_\d+\b")



def extract_meeting_id(zitting: str) -> str:
    """Extract a 4-part Ghent meeting ID (e.g. '25.1218.5714.0722') from a URL or raw ID."""
    match = _MEETING_ID_RE.search(zitting.strip())
    if match:
        return match.group(1)
    raise ValueError(
        f"Could not extract a valid Ghent meeting ID from {zitting!r}. "
        "Expected format 'YY.MMDD.XXXX.XXXX' or a URL containing '/zittingen/YY.MMDD.XXXX.XXXX'."
    )


def parse_zitting_agendapunten(
    html: str,
    meeting_id: str = "",
    *,
    only_published: bool = True,
) -> list[dict[str, Any]]:
    """Extract agenda items from City of Ghent meeting (zitting) HTML."""
    soup = BeautifulSoup(html, "html.parser")
    if not meeting_id:
        link = soup.find(
            "a", href=re.compile(r"/zittingen/([0-9]{2}\.[0-9]{4}\.[0-9]{4}\.[0-9]{4})")
        )
        if link and link.get("href"):
            match = _MEETING_ID_RE.search(link["href"])
            if match:
                meeting_id = match.group(1)

    items: list[dict[str, Any]] = []
    for li in soup.find_all("li", class_="agendapage"):
        item_id = li.get("data-meeting-item-id") or ""
        pos_el = li.find(class_="position")
        pos_text = pos_el.get_text(strip=True) if pos_el else ""
        position = int(pos_text) if pos_text.isdigit() else None

        title_el = li.find("a", class_="title")
        title = title_el.get_text(strip=True) if title_el else ""
        href = title_el.get("href") if title_el else None

        item_div = li.find(class_="item")
        item_classes = item_div.get("class", []) if item_div else []
        is_published = bool(href) and ("not-published" not in item_classes)

        if only_published and not is_published:
            continue

        item_url = (
            f"{BASE_URL}{href}"
            if href
            else (agenda_item_url(meeting_id, item_id) if is_published else None)
        )

        doc_match = _DOCUMENT_ID_RE.search(title)
        document_id = doc_match.group(0) if doc_match else None

        summary_el = li.find(class_="summary")
        summary = (
            summary_el.get_text(separator=" ", strip=True) if summary_el else None
        )

        items.append(
            {
                "agenda_item_id": item_id,
                "meeting_id": meeting_id,
                "position": position,
                "title": title,
                "url": item_url,
                "is_published": is_published,
                "document_id": document_id,
                "summary": summary,
            }
        )

    return items


async def fetch_agendapunten_urls(
    url: str = "https://ebesluitvorming.gent.be/zittingen",
    *,
    month: int | str | None = None,
    year: int | str | None = None,
    client: httpx.AsyncClient | None = None,
) -> list[str]:
    """Fetch all published agendapunten URLs from https://ebesluitvorming.gent.be/zittingen.

    Parameters
    ----------
    url
        Either "https://ebesluitvorming.gent.be/zittingen" (or "/zittingen/lijst") to discover
        all meetings in the overview, OR a specific meeting URL/ID to scrape only that meeting.
    month, year
        Optional month and year when scraping the zittingen list (e.g. month=8, year=2026).
    client
        Optional httpx.AsyncClient to reuse.

    Returns
    -------
    list[str]
        Public URLs of all published agenda items found.
    """
    own_client = client is None
    c = client or httpx.AsyncClient(
        timeout=30,
        follow_redirects=True,
        headers={"User-Agent": "ipi-legislation/0.1 (+research)"},
    )
    try:
        meeting_match = _MEETING_ID_RE.search(url.strip())
        if meeting_match:
            meeting_id = meeting_match.group(1)
            meeting_url = f"{BASE_URL}/zittingen/{meeting_id}"
            res = await c.get(meeting_url)
            res.raise_for_status()
            soup = BeautifulSoup(res.text, "html.parser")
            urls = []
            for li in soup.find_all("li", class_="agendapage"):
                title_el = li.find("a", class_="title")
                item_div = li.find(class_="item")
                is_pub = (
                    title_el
                    and title_el.get("href")
                    and ("not-published" not in (item_div.get("class", []) if item_div else []))
                )
                if is_pub:
                    urls.append(f"{BASE_URL}{title_el['href']}")
            return urls

        target_url = url.strip()
        if target_url.rstrip("/").endswith("/zittingen"):
            target_url = f"{BASE_URL}/zittingen/lijst"
        if month and year and "month=" not in target_url:
            sep = "&" if "?" in target_url else "?"
            target_url = f"{target_url}{sep}month={int(month):02d}&year={year}"

        res = await c.get(target_url)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        meeting_hrefs = list(
            dict.fromkeys(
                a["href"]
                for a in soup.find_all("a", href=True)
                if a["href"].startswith("/zittingen/")
                and a["href"] not in ("/zittingen/lijst", "/zittingen/kalender")
                and "?" not in a["href"]
            )
        )

        async def _fetch_meeting(href: str) -> list[str]:
            try:
                m_res = await c.get(f"{BASE_URL}{href}")
                if m_res.status_code != 200:
                    return []
                m_soup = BeautifulSoup(m_res.text, "html.parser")
                m_urls = []
                for li in m_soup.find_all("li", class_="agendapage"):
                    title_el = li.find("a", class_="title")
                    item_div = li.find(class_="item")
                    is_pub = (
                        title_el
                        and title_el.get("href")
                        and ("not-published" not in (item_div.get("class", []) if item_div else []))
                    )
                    if is_pub:
                        m_urls.append(f"{BASE_URL}{title_el['href']}")
                return m_urls
            except Exception:
                return []

        all_results = await asyncio.gather(*[_fetch_meeting(h) for h in meeting_hrefs])
        return [item_url for sub in all_results for item_url in sub]
    finally:
        if own_client:
            await c.aclose()


def scrape_agendapunten(
    zitting: str = "https://ebesluitvorming.gent.be/zittingen",
    *,
    month: int | str | None = None,
    year: int | str | None = None,
    only_published: bool = True,
    urls_only: bool = False,
    client: httpx.Client | None = None,
) -> list[dict[str, Any]] | list[str]:
    """Scrape agendapunten from a City of Ghent meeting or zittingen overview."""
    stripped = zitting.strip()
    if stripped.startswith("<"):
        items = parse_zitting_agendapunten(stripped, only_published=only_published)
    elif stripped.rstrip("/").endswith("/zittingen") or "/zittingen/lijst" in stripped:
        urls = asyncio.run(fetch_agendapunten_urls(stripped, month=month, year=year))
        if urls_only:
            return urls
        return [{"url": u} for u in urls]
    else:
        meeting_id = extract_meeting_id(stripped)
        url = f"{BASE_URL}/zittingen/{meeting_id}"
        own_client = client is None
        http_client = client or httpx.Client(
            timeout=30,
            follow_redirects=True,
            headers={"User-Agent": "ipi-legislation/0.1 (+research)"},
        )
        try:
            response = http_client.get(url)
            response.raise_for_status()
            html = response.text
        finally:
            if own_client:
                http_client.close()

        items = parse_zitting_agendapunten(
            html, meeting_id=meeting_id, only_published=only_published
        )

    if urls_only:
        return [it["url"] for it in items if it.get("url")]
    return items


async def async_scrape_agendapunten(
    zitting: str = "https://ebesluitvorming.gent.be/zittingen",
    *,
    month: int | str | None = None,
    year: int | str | None = None,
    only_published: bool = True,
    urls_only: bool = False,
    client: httpx.AsyncClient | None = None,
) -> list[dict[str, Any]] | list[str]:
    """Asynchronously scrape agendapunten from a City of Ghent meeting or zittingen overview."""
    stripped = zitting.strip()
    if stripped.startswith("<"):
        items = parse_zitting_agendapunten(stripped, only_published=only_published)
    elif stripped.rstrip("/").endswith("/zittingen") or "/zittingen/lijst" in stripped:
        urls = await fetch_agendapunten_urls(
            stripped, month=month, year=year, client=client
        )
        if urls_only:
            return urls
        return [{"url": u} for u in urls]
    else:
        meeting_id = extract_meeting_id(stripped)
        url = f"{BASE_URL}/zittingen/{meeting_id}"
        own_client = client is None
        http_client = client or httpx.AsyncClient(
            timeout=30,
            follow_redirects=True,
            headers={"User-Agent": "ipi-legislation/0.1 (+research)"},
        )
        try:
            response = await http_client.get(url)
            response.raise_for_status()
            html = response.text
        finally:
            if own_client:
                await http_client.aclose()

        items = parse_zitting_agendapunten(
            html, meeting_id=meeting_id, only_published=only_published
        )

    if urls_only:
        return [it["url"] for it in items if it.get("url")]
    return items


In [8]:
import asyncio
from tqdm import tqdm

processed = []


async def link_reference(reference, *, verbose: bool = False):
    document_context = DocumentContext(
        municipality="Ghent, Flanders, Belgium",
        description=reference.context,
    )
    target_reference = reference.doc_reference or (
        reference.mentions[0] if reference.mentions else ""
    )
    results, cache = await find_link_to_document_cached(
        target_reference,
        document_context=document_context,
        doc_reference=reference.doc_reference,
        reference_class=reference.reference_class,
        verbose=verbose,
    )
    return {
        "reference": reference,
        "results": results,
        "cache": cache,
    }


In [9]:
# overview of the grouped references and their resolution results
resolved_count = sum(
    1 for item in processed if item["results"].status.value == "resolved"
)
print(
    f"Successfully resolved {resolved_count} out of "
    f"{len(processed)} document references."
)


for item in processed:
    reference = item["reference"]
    response = item["results"]
    print("\n---------")
    print(
        f"Item: {response.classification.document_type}|"
        f"{response.classification.jurisdiction}|"
        f"{response.classification.source}"
    )
    print(f"Document: {reference.doc_reference}")
    print(f"Cache: {item.get('cache', {}).get('status', 'not_checked')}")
    print(f"Context: {reference.context}")
    print(f"Mentions: {reference.mentions}")
    print(f"Status: {response.status.value}")
    if response.status.value != "resolved":
        print(f"Reasoning: {response.reason}")

Successfully resolved 6 out of 6 document references.

---------
Item: local_decision|local|ebesluitvorming
Document: Gentse gemeenteraadsbeslissing 2023_GR_01131 van 19 december 2023, tot wijziging van het retributiereglement voor een lage-emissiezone
Cache: miss
Context: Beslissing van de Gentse gemeenteraad tot wijziging van het retributiereglement voor de lage-emissiezone in Gent.
Mentions: ['2023_GR_01131 - Retributiereglement voor een lage-emissiezone (LEZ) in Gent - Wijziging', '2023_GR_01131 - Retributiereglement voor een lage-emissiezone (LEZ) in Gent - Wijziging', '2023_GR_01131 - Retributiereglement voor een lage-emissiezone (LEZ) in Gent - Wijziging', '2023_GR_01131 - Retributiereglement voor een lage-emissiezone (LEZ) in Gent - Wijziging']
Status: resolved

---------
Item: local_decision|local|ebesluitvorming
Document: Retributiereglement voor een lage-emissiezone (LEZ) in Gent
Cache: hit
Context: Gentse gemeentelijke regeling voor retributies verbonden aan de lage-emissie

In [10]:


output_file = Path("agenda_item_linked-{}.html".format(url.split("/")[-1]))
write_linked_html(extracted["html"], processed, output_file)

print(f"Linked HTML written to {output_file}")

Linked HTML written to agenda_item_linked-23.1116.3224.1663.html


In [ ]:
# [TextReference(text='Het Decreet over het lokaal bestuur van 22 december 2017, artikel 63', reference_class='decree'),
#  TextReference(text='De Nieuwe gemeentewet, artikelen 133 en 135, Â§ 2, 3Â°', reference_class='law'),
#  TextReference(text='Het politiereglement op de openbare rust en veiligheid, goedgekeurd in de gemeenteraad van 19 januari 1998, artikel 1, Â§ 1', reference_class='municipal police regulation'),
#  TextReference(text='De gecoÃ¶rdineerde Grondwet, artikel 26', reference_class='constitution'),
#  TextReference(text='Het politiereglement op de openbare rust en veiligheid, goedgekeurd in de gemeenteraad van 19 januari 1998, artikel 1', reference_class='municipal police regulation'),
#  TextReference(text='artikel 19, tweede lid van de gecoÃ¶rdineerde wetten op de Raad van State', reference_class='law'),
#  TextReference(text='De Algemene Verordening Gegevensbescherming', reference_class='European regulation'),
#  TextReference(text='â€˜codex politiereglementen Stad Gent gesanctioneerd met een gemeentelijke administratieve sanctieâ€™', reference_class='municipal code'),
#  TextReference(text='het politiereglement op de openbare rust en de veiligheid (Goedgekeurd in de gemeenteraad van 19 januari 1998)', reference_class='municipal police regulation'),
#  TextReference(text='de Wet van 02 oktober 2017 tot regeling van de private en bijzondere veiligheid', reference_class='law'),
#  TextReference(text='de omzendbrief SPV07 van 29 maart 2018', reference_class='circular'),
#  TextReference(text='wet 28 december 1983', reference_class='law')]

# process

In [9]:
# Fetch all published agendapunten URLs starting from the zittingen overview
# (e.g. for August 2026, which contains zitting 25.1218.5714.0722, or omit month to get the current month)
urls = await fetch_agendapunten_urls("https://ebesluitvorming.gent.be/zittingen", month=8, year=2026)

print(f"Total published agendapunten URLs found: {len(urls)}")

selected_urls = urls[140:200]
total_items = len(selected_urls)

for idx, url in enumerate(selected_urls, start=1):
    item_id = url.split("/")[-1]
    print(f"\n{'=' * 80}")
    print(f"[{idx}/{total_items}] Fetched from URL: {url}")

    extracted = await extract_references(url)
    references = extracted.get("references", [])
    print(f"Extracted {len(references)} document references")

    tmp_file = Path(f"results/agenda_item_linked-{item_id}-tmp.html")
    write_linked_html(extracted["html"], references, tmp_file)

    print("Matching results:")
    processed = []
    for ref in references:
        item = await link_reference(ref, verbose=False)
        processed.append(item)
        cache_status = item.get("cache", {}).get("status", "miss").upper()
        lookup = item.get("cache", {}).get("lookup", "no_match")
        res = item.get("results")
        status_val = res.status.value if res else "unresolved"
        result_url = res.results[0].url if res and res.results else "no link"
        print(f"  [CACHE {cache_status}] {ref.doc_reference!r} lookup={lookup} -> {status_val}: {result_url}")

    output_file = Path(f"results/agenda_item_linked-{item_id}.html")
    write_linked_html(extracted["html"], processed, output_file)

    print(f"Saved linked HTML to: {output_file}")


Total published agendapunten URLs found: 526

[1/60] Fetched from URL: https://ebesluitvorming.gent.be/zittingen/26.0126.0809.8993/agendapunten/26.0810.1720.5868
Extracted 14 document references
Matching results:
  [CACHE MISS] 'Besluit van het college van burgemeester en schepenen van Gent van 20 augustus 2026, dossier 2026_CBS_06624 / OMV_2026070977' lookup=no_match -> resolved: https://ebesluitvorming.gent.be/zittingen/26.0126.0809.8993/agendapunten/26.0810.1720.5868
  [CACHE HIT] 'Decreet betreffende de omgevingsvergunning van 25 april 2014' lookup=title_match -> resolved: https://codex.vlaanderen.be/Zoeken/Document.aspx?DID=1024690
  [CACHE MISS] 'Omgevingsvergunning OMV_2020011285 van 17 september 2020' lookup=no_match -> resolved: https://ebesluitvorming.gent.be/zittingen/19.0917.4180.0523/agendapunten/20.0910.5014.7450
  [CACHE HIT] 'Gemeentelijk RUP ANTWERPSESTEENWEG - ORCHIDEESTRAAT, definitief vastgesteld door de Gentse gemeenteraad op 26 juni 2018' lookup=title_match -> res

CancelledError: 

# testing

In [3]:

ghent_document_context = DocumentContext(
    municipality="Ghent",
    description=(
        "Extracted from documents from Ghent. Use Ghent, Flemish regional law, "
        "and Belgian federal law as fallback context only."
    ),
)


results = await find_link_to_document("Reglement 'Belasting op film- en erotische voorstellingen' van de stad Gent, zoals gewijzigd met ingang van 1 januari 2026",
                                      document_context=ghent_document_context)
format_resolution(results)

RESOLUTION SUMMARY
Reference : Reglement 'Belasting op film- en erotische voorstellingen' van de stad Gent, zoals gewijzigd met ingang van 1 januari 2026
Document Class : regulation [local]
Status    : resolved
Reason    : Candidate 0 is the sole candidate returned by the official reglement searches and matches the requested official title exactly. It is classified as a Belastingreglement approved by the Gemeenteraad on 24 February 2026, which is consistent with a regulation amended with effect from 1 January 2026. No competing candidate was found. The supplied candidate metadata does not separately state the municipality, but the source context and request identify the regulation as belonging to the city of Gent.
Attempts  : 3
Results   : 1

Result 1
  Title       : Belasting op film- en erotische voorstellingen
  Source      : reglement
  Document ID : 803320
  URL         : https://stad.gent/nl/reglementen/belasting-op-film-en-erotische-voorstellingen
  Retrieved   : 2026-09-28T09:4

In [4]:
ghent_document_context = DocumentContext(
    municipality="Ghent",
    description=(
        "Extracted from documents from Ghent. Use Ghent, Flemish regional law, "
        "and Belgian federal law as fallback context only."
    ),
)


results = await find_link_to_document("De Algemene Verordening Gegevensbescherming",
                                      document_context=ghent_document_context)
format_resolution(results)

RESOLUTION SUMMARY
Reference : De Algemene Verordening Gegevensbescherming
Document Class : regulation [european]
Status    : resolved
Reason    : Kandidaat 0 is de officiële EUR-Lex-verordening en identificeert exact Verordening (EU) 2016/679 van het Europees Parlement en de Raad van 27 april 2016, bekend als de Algemene Verordening Gegevensbescherming. De overige kandidaten zijn informatiepagina’s, een verwijzende verordening, een nieuwsbericht of een geconsolideerde weergave en vormen niet een afzonderlijk basism instrument.
Attempts  : 1
Results   : 1

Result 1
  Title       : Verordening - 2016/679 - EN - avg - EUR-Lex
  Source      : web_search
  Document ID : 32016R0679
  URL         : https://eur-lex.europa.eu/legal-content/NL/TXT/?uri=CELEX:32016R0679
  Retrieved   : 2026-09-28T09:48:11.905134+00:00
  Metadata    : {'snippet': 'Verordening (EU) 2016/679 van het Europees Parlement en de Raad van 27 april 2016 betreffende de bescherming van natuurlijke personen in verband met de

In [5]:
ghent_document_context = DocumentContext(
    municipality="Ghent",
    description=(
        "Extracted from documents from Ghent. Use Ghent, Flemish regional law, "
        "and Belgian federal law as fallback context only."
    ),
)

results = await find_link_to_document(
    "wet 28 december 1983",
    document_context=ghent_document_context,
)
format_resolution(results)

RESOLUTION SUMMARY
Reference : wet 28 december 1983
Document Class : law [federal]
Status    : resolved
Reason    : Candidate 7 is the base instrument matching the reference: the Belgian federal law dated 28 December 1983, officially titled “Wet betreffende de vergunning voor het verstrekken van sterke drank.” Candidates 8 and 9 are later amendment laws that cite or modify the 1983 law, not the requested base instrument. Candidate 7 has identifier 1983004493.
Attempts  : 2
Results   : 1

Result 1
  Title       : Wet betreffende de vergunning voor het verstrekken van sterke ...
  Source      : web_search
  Document ID : 1983004493
  URL         : https://www.remi.legal/nl/wetgeving/1983004493
  Retrieved   : 2026-09-28T09:48:35.334994+00:00
  Metadata    : {'snippet': '28 DECEMBER 1983. - [Wet betreffende de vergunning voor het verstrekken van sterke drank.] Zoek in deze wet Lees vanaf begin Toon inhoud'}


In [6]:


ghent_document_context = DocumentContext(
    municipality="Ghent",
    description=(
        "Extracted from documents from Ghent. Use Ghent, Flemish regional law, "
        "and Belgian federal law as fallback context only."
    ),
)

results = await find_link_to_document(
    "politiereglement betreffende de organisatie van activiteiten tijdens het wereldkampioenschap voetbal 2026",
    document_context=ghent_document_context,
)
format_resolution(results)

RESOLUTION SUMMARY
Reference : politiereglement betreffende de organisatie van activiteiten tijdens het wereldkampioenschap voetbal 2026
Document Class : regulation [local]
Status    : resolved
Reason    : Candidate 0 is confirmed: its official title exactly matches the reference, it is a Politiereglement from the reglement source, and it concerns the stated local activity-organization subject. The candidate is approved by the Gemeenteraad, consistent with a Ghent municipal regulation. No competing candidates were returned.
Attempts  : 1
Results   : 1

Result 1
  Title       : Politiereglement betreffende de organisatie van activiteiten tijdens het wereldkampioenschap voetbal 2026
  Source      : reglement
  Document ID : 832438
  URL         : https://stad.gent/nl/reglementen/politiereglement-betreffende-de-organisatie-van-activiteiten-tijdens-het-wereldkampioenschap-voetbal-1
  Retrieved   : 2026-09-28T09:48:50.016159+00:00
  Metadata    : {'type': 'Politiereglement', 'approval_body'

In [7]:


ghent_document_context = DocumentContext(
    municipality="Ghent",
    description=(
        "Extracted from documents from Ghent. Use Ghent, Flemish regional law, "
        "and Belgian federal law as fallback context only."
    ),
)

results = await find_link_to_document(
    "De Nieuwe Gemeentewet, artikel 119",
    document_context=ghent_document_context,
)
format_resolution(results)

RESOLUTION SUMMARY
Reference : De Nieuwe Gemeentewet, artikel 119
Document Class : law [federal]
Status    : resolved
Reason    : Candidate 0 matches the requested base law by title, document kind, and NUMAC identifier: “Nieuwe Gemeentewet” (1988062450). The reference specifies the federal law and does not identify the Brussels-Capital Region; candidate 1 is explicitly the distinct Brussels regional version. Article 119 is an article within the base instrument, not an amendment document.
Attempts  : 2
Results   : 1

Result 1
  Title       : Nieuwe Gemeentewet | Remi
  Source      : web_search
  Document ID : 1988062450
  URL         : https://www.remi.legal/nl/wetgeving/1988062450
  Retrieved   : 2026-09-28T09:49:01.207060+00:00
  Metadata    : {'snippet': 'Nieuwe Gemeentewet · NUMAC 1988062450. Lees en doorzoek Nieuwe Gemeentewet: geconsolideerde tekst, artikels, structuur en documentinformatie uit...'}


In [11]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException

query = "32016R0679"

websites = {
    "ejustice.just.fgov.be",
    "eur-lex.europa.eu",
    "www.remi.legal"
}

raw_results = []
for website in sorted(websites):
    try:
        site_results = DDGS().text(
            f"{query} site:{website}",
            region="be-nl",
            max_results=20,
            backend="yahoo",
        )
    except DDGSException:
        continue
    raw_results.extend(site_results)

raw_results

[{'title': 'Document 32016R0679',
  'href': 'https://eur-lex.europa.eu/eli/reg/2016/679/oj/eng',
  'body': 'Jul 9, 2015 Â· Document 32016R0679 Regulation (EU) 2016/679 of the European Parliament and of the Council of 27 April 2016 on the protection of natural persons with regard to the processing of personal data and on the free movement of such data, and repealing Directive 95/46/EC (General Data Protection Regulation) (Text with EEA relevance)'},
 {'title': 'Regulation - 2016/679 - EN - gdpr - EUR-Lex',
  'href': 'https://eur-lex.europa.eu/legal-content/EN/HIS/?uri=oj:JOL_2016_119_R_0001',
  'body': 'Document 32016R0679 Procedure 2012/0011/COD COM (2012) 11: Proposal for a REGULATION OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL on the protection of individuals with regard to the processing of personal data and on the free movement of such data (General Data Protection Regulation)'},
 {'title': 'EU law - EUR-Lex',
  'href': 'https://eur-lex.europa.eu/homepage.html?locale=EN',
  'body

In [8]:
path_dataset = "dataset\\data\\mentions_gent_2026-03-23_gr.jsonl"

dataset = [json.loads(line) for line in open(path_dataset, encoding="utf-8")]

In [12]:
resolved = [entry for entry in dataset if "resolved" == entry["resolution_status"]]

In [30]:
resolved[0]

{'block_index': 0,
 'classification_rule': 'canonical_law_titles_v1',
 'confidence': 0.97,
 'decision_date': 'ma 23/03/2026 - 19:08',
 'extraction_method': 'explicit_legal_basis_section',
 'governing_body': 'gemeenteraad',
 'mention': 'De Nieuwe Gemeentewet, artikel 119;',
 'mention_html': '<li>De Nieuwe Gemeentewet, artikel 119;</li>',
 'mention_id': 'gent:92ff31acdc8231d2fd5c3624',
 'municipality': 'Gent',
 'predicted_class': 'law',
 'resolution_status': 'resolved',
 'resolved_documents': [{'confidence': 1.0,
   'document_id': '1009730',
   'match_method': 'canonical_title_verified_in_live_codex_search',
   'source': 'codex',
   'source_record_id': '1009730',
   'title': 'GecoÃ¶rdineerde Wet Nieuwe Gemeentewet',
   'url': 'https://codex.vlaanderen.be/Zoeken/Document.aspx?DID=1009730'}],
 'retrieved_at': '2026-08-01T11:44:41.058454+00:00',
 'section': 'Regelgeving waaruit blijkt dat het orgaan bevoegd is',
 'section_type': 'legal_authority',
 'session_date': '2026-03-23T19:00:00.000+0